# 01 — Bronze (VARIANT) — API v1

Generates synthetic NHL play-by-play events and lands them in `bronze.plays_raw` with the raw API payload stored as **VARIANT**.

**v1 payload keys:**

```
game_id, season, period, period_time,
event_type, team_abbrev, player_id,
x_coord, y_coord, event_ts
```

Bronze schema (table columns):

| Column | Type | Notes |
|---|---|---|
| `ingest_ts` | `timestamp` | when the row landed |
| `source_version` | `string` | `v1` or `v2` — lets us tell which API revision delivered the row |
| `payload` | `variant` | the raw JSON event |

Storing the payload as VARIANT means we can land *any* future API revision without an upfront schema change.

In [0]:
%pip install dotenv

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os, json, random, datetime as dt
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'hockey_schema_monitoring')
BRONZE_TABLE  = f'{UC_CATALOG}.{UC_SCHEMA}_bronze.plays_raw'

N_GAMES         = int(os.getenv('N_GAMES', '8'))
EVENTS_PER_GAME = int(os.getenv('EVENTS_PER_GAME', '400'))
SEED            = int(os.getenv('RANDOM_SEED', '42'))

print(f'Target table : {BRONZE_TABLE}')
print(f'Games        : {N_GAMES}')
print(f'Events/game  : {EVENTS_PER_GAME}')

Target table : alexander_booth.hockey_schema_monitoring_bronze.plays_raw
Games        : 8
Events/game  : 400


In [0]:
TEAMS  = ['TOR', 'BOS', 'NYR', 'EDM', 'COL', 'VGK', 'FLA', 'CAR', 'TBL', 'PIT']
EVENTS = ['shot-on-goal', 'goal', 'hit', 'faceoff', 'giveaway', 'takeaway',
          'blocked-shot', 'missed-shot', 'penalty']

rng = random.Random(SEED)

def make_play(game_id: str, season: str, period: int, t_sec: int) -> dict:
    mm, ss = divmod(20 * 60 - t_sec, 60)
    return {
        'game_id'     : game_id,
        'season'      : season,
        'period'      : period,
        'period_time' : f'{mm:02d}:{ss:02d}',
        'event_type'  : rng.choice(EVENTS),
        'team_abbrev' : rng.choice(TEAMS),
        'player_id'   : 8470000 + rng.randint(0, 99999),
        'x_coord'     : rng.randint(-100, 100),
        'y_coord'     : rng.randint(-42, 42),
        'event_ts'    : (dt.datetime(2026, 5, 20, 19, 0)
                          + dt.timedelta(seconds=t_sec + (period - 1) * 20 * 60)
                       ).isoformat() + 'Z',
    }

rows = []
for g in range(N_GAMES):
    game_id = f'202602{1000 + g:04d}'
    for n in range(EVENTS_PER_GAME):
        period = 1 + n // (EVENTS_PER_GAME // 3)
        t_sec  = rng.randint(0, 20 * 60 - 1)
        rows.append({
            'source_version': 'v1',
            'payload_json'  : json.dumps(make_play(game_id, '20252026', min(period, 3), t_sec)),
        })

print(f'Generated {len(rows):,} events across {N_GAMES} games')
print('Sample payload:')
print(json.dumps(json.loads(rows[0]['payload_json']), indent=2))

Generated 3,200 events across 8 games
Sample payload:
{
  "game_id": "2026021000",
  "season": "20252026",
  "period": 1,
  "period_time": "16:12",
  "event_type": "shot-on-goal",
  "team_abbrev": "COL",
  "player_id": 8502098,
  "x_coord": -43,
  "y_coord": -25,
  "event_ts": "2026-05-20T19:03:48Z"
}


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Build a small string-only DataFrame, then turn payload_json into VARIANT via parse_json.
# Doing the parse server-side keeps the variant materialization in Spark, not in the driver.
schema = StructType([
    StructField('source_version', StringType(), False),
    StructField('payload_json',   StringType(), False),
])
df = (spark.createDataFrame(rows, schema=schema)
           .withColumn('ingest_ts', F.current_timestamp())
           .withColumn('payload',   F.expr('parse_json(payload_json)'))
           .select('ingest_ts', 'source_version', 'payload'))

(df.write
   .mode('overwrite')
   .option('overwriteSchema', 'true')
   .saveAsTable(BRONZE_TABLE))

print(f'Wrote {df.count():,} rows to {BRONZE_TABLE}')

Wrote 3,200 rows to alexander_booth.hockey_schema_monitoring_bronze.plays_raw


In [0]:
spark.sql(f'DESCRIBE TABLE {BRONZE_TABLE}').show(truncate=False)

print('\nSample rows (note payload is VARIANT — dot-paths work in SQL):')
spark.sql(f'''
    SELECT
        source_version,
        payload:game_id::string     AS game_id,
        payload:event_type::string  AS event_type,
        payload:team_abbrev::string AS team,
        payload:x_coord::int        AS x,
        payload:y_coord::int        AS y
    FROM {BRONZE_TABLE}
    LIMIT 5
''').show(truncate=False)

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|ingest_ts     |timestamp|NULL   |
|source_version|string   |NULL   |
|payload       |variant  |NULL   |
+--------------+---------+-------+


Sample rows (note payload is VARIANT — dot-paths work in SQL):
+--------------+----------+------------+----+---+---+
|source_version|game_id   |event_type  |team|x  |y  |
+--------------+----------+------------+----+---+---+
|v1            |2026021006|blocked-shot|CAR |-38|-30|
|v1            |2026021006|hit         |VGK |2  |-1 |
|v1            |2026021006|blocked-shot|PIT |36 |-33|
|v1            |2026021006|missed-shot |PIT |-44|35 |
|v1            |2026021006|faceoff     |VGK |-13|22 |
+--------------+----------+------------+----+---+---+

